# T12 CMU ORM — Final Test Inference

이 노트북은 `test_data.csv`를 입력으로 받아 **`test_submission.csv`**를 생성합니다.

동결된 T12 파이프라인은 다음과 같습니다.

1. `Qwen/Qwen2.5-3B-Instruct`로 문항당 후보 풀이 32개 생성
2. 함께 제공된 LoRA ORM으로 각 후보 풀이의 정답 가능성 채점
3. 정수 답별로 후보를 묶고 `지지 표 수 × ORM 점수의 기하평균` 계산
4. 최고 가중치 답을 제출 정수로 선택

정답 열은 읽지 않습니다. 생성과 채점은 별도 프로세스에서 실행되므로 두 단계 사이에 GPU 메모리가 완전히 반환되며, 중간 JSONL을 이용해 중단 후 재실행할 수 있습니다.

## 1. Setup

먼저 아래 파라미터를 확인합니다. 기본값은 이 폴더 안의 `test_data.csv` 전체 2,000문항을 처리합니다. `LIMIT`는 개발용 스모크 테스트에만 사용하고 최종 제출 시 반드시 `None`으로 둡니다.

베이스 모델은 Hugging Face에서 고정 revision을 다운로드합니다. 오프라인 환경에서는 `BASE_MODEL_ID_OR_PATH`를 동일 revision의 로컬 snapshot 경로로 바꾸세요.

In [ ]:
from pathlib import Path
import csv
import importlib.util
import json
import os
import subprocess
import sys

# 노트북을 final/에서 열거나 저장소 루트에서 열어도 경로를 찾도록 합니다.
if Path("t12_pipeline.py").is_file():
    PACKAGE_DIR = Path.cwd().resolve()
elif Path("final/t12_pipeline.py").is_file():
    PACKAGE_DIR = (Path.cwd() / "final").resolve()
else:
    raise FileNotFoundError("t12_pipeline.py가 있는 final 폴더를 찾을 수 없습니다.")

INPUT_PATH = PACKAGE_DIR / "test_data.csv"
ADAPTER_PATH = PACKAGE_DIR / "model" / "t12_cmu_orm_adapter"
WORK_DIR = PACKAGE_DIR / "output"
OUTPUT_PATH = PACKAGE_DIR / "test_submission.csv"

BASE_MODEL_ID_OR_PATH = "Qwen/Qwen2.5-3B-Instruct"
BASE_MODEL_REVISION = "aa8e72537993ba99e69dfaafa59ed015b17504d1"

# 최종 실행은 None. 예: LIMIT=2는 파이프라인 점검용이며 최종 제출 파일이 아닙니다.
LIMIT = None
GPU_MEMORY_UTILIZATION = 0.92
MAX_NUM_SEQS = 256
GENERATION_GPU = "0"
SCORING_GPU = "0"

print("package:", PACKAGE_DIR)
print("input:", INPUT_PATH)
print("output:", OUTPUT_PATH)

### Dependencies

CUDA용 PyTorch는 실행 장비에 맞는 빌드가 설치되어 있어야 합니다. 나머지 패키지가 없다면 `INSTALL_DEPENDENCIES=True`로 한 번 실행한 뒤 커널을 재시작하세요.

In [ ]:
INSTALL_DEPENDENCIES = False

if INSTALL_DEPENDENCIES:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(PACKAGE_DIR / "requirements.txt")],
        check=True,
    )

required_modules = ["torch", "transformers", "peft", "safetensors", "vllm"]
missing = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(
        f"필수 패키지가 없습니다: {missing}. INSTALL_DEPENDENCIES=True로 실행한 뒤 커널을 재시작하세요."
    )

import torch
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU가 필요합니다.")
print("torch:", torch.__version__)
print("visible CUDA devices:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print(index, torch.cuda.get_device_name(index))

## 2. Package and input checks

입력 ID·질문, 비어 있는 라벨 열, LoRA 종류·rank, 베이스 모델 identity, 457 MiB adapter SHA-256을 확인합니다. 이 검사는 모델을 GPU에 올리지 않습니다.

In [ ]:
def stage_command(stage):
    command = [
        sys.executable,
        str(PACKAGE_DIR / "t12_pipeline.py"),
        stage,
        "--input", str(INPUT_PATH),
        "--adapter", str(ADAPTER_PATH),
        "--work-dir", str(WORK_DIR),
        "--output", str(OUTPUT_PATH),
        "--base-model", BASE_MODEL_ID_OR_PATH,
        "--revision", BASE_MODEL_REVISION,
        "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
        "--max-num-seqs", str(MAX_NUM_SEQS),
    ]
    if LIMIT is not None:
        command.extend(["--limit", str(LIMIT)])
    return command

validation = subprocess.run(
    stage_command("validate"), check=True, text=True, capture_output=True
)
print(validation.stdout)

## 3. Generate 32 candidate solutions per question

순정 Qwen2.5-3B-Instruct로 후보를 생성합니다. 동결 설정은 `seed=42`, `temperature=0.8`, `top_p=0.95`, `max_new_tokens=2048`, `k=32`입니다.

결과는 `output/generations.jsonl`에 문항 단위로 누적됩니다. 셀이 중단되면 그대로 다시 실행할 수 있습니다. 한 문항의 32개 출력 중 일부만 기록된 비정상 파일은 안전을 위해 자동 재개하지 않고 오류로 중단합니다.

In [ ]:
generation_env = os.environ.copy()
generation_env["CUDA_VISIBLE_DEVICES"] = GENERATION_GPU
generation_env["VLLM_BATCH_INVARIANT"] = "1"
generation_env["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
subprocess.run(stage_command("generate"), check=True, env=generation_env)

## 4. Score candidates with the trained ORM

생성 프로세스가 종료된 뒤 같은 고정 Qwen base를 `AutoModelForSequenceClassification(num_labels=1)`로 불러오고, `model/t12_cmu_orm_adapter`의 LoRA 및 scalar score head를 결합합니다.

원래 T12와 동일하게 BF16, 최대 길이 4096, batch 4, 128-token 고정 padding bucket을 사용합니다. 결과는 `output/candidate_scores.jsonl`에 1,024개 단위로 저장되어 재실행할 수 있습니다.

In [ ]:
scoring_env = os.environ.copy()
scoring_env["CUDA_VISIBLE_DEVICES"] = SCORING_GPU
subprocess.run(stage_command("score"), check=True, env=scoring_env)

## 5. Aggregate and write `test_submission.csv`

후보 텍스트에서 모델이 직접 쓴 정수만 추출합니다. 계산·수식 평가·정답 참조는 하지 않습니다. 유효 후보가 전혀 없는 문항은 기존 T12 제출 계약과 동일하게 `0`으로 폴백합니다.

In [ ]:
subprocess.run(stage_command("submit"), check=True)

with OUTPUT_PATH.open("r", encoding="utf-8-sig", newline="") as handle:
    reader = csv.DictReader(handle)
    submission_rows = list(reader)

print("submission rows:", len(submission_rows))
print("output:", OUTPUT_PATH)
print("first five rows:")
for row in submission_rows[:5]:
    print(row)

## 6. Final checks

입력 순서, 행 수, ID 고유성, 빈 값, 정수 형식을 마지막으로 확인합니다. `LIMIT`가 `None`일 때만 완전한 최종 제출 파일입니다.

In [ ]:
import re

with INPUT_PATH.open("r", encoding="utf-8-sig", newline="") as handle:
    input_reader = csv.DictReader(handle)
    input_id_column = next(name for name in input_reader.fieldnames if name.strip().casefold() == "id")
    input_ids = [row[input_id_column].strip() for row in input_reader]

submission_id_column = next(name for name in submission_rows[0] if name.strip().casefold() == "id")
submission_ids = [row[submission_id_column].strip() for row in submission_rows]
integer_pattern = re.compile(r"^-?(?:0|[1-9][0-9]*)$")

expected_ids = input_ids if LIMIT is None else input_ids[:LIMIT]
assert submission_ids == expected_ids, "입력 ID 순서 또는 범위가 다릅니다."
assert len(submission_ids) == len(set(submission_ids)), "중복 ID가 있습니다."
assert all(integer_pattern.fullmatch(row["answer"] or "") for row in submission_rows), "정수가 아닌 답이 있습니다."
if LIMIT is None:
    assert len(submission_rows) == len(input_ids), "최종 제출 행 수가 입력과 다릅니다."
else:
    print("주의: LIMIT가 설정된 스모크 결과이므로 최종 제출용이 아닙니다.")

with (WORK_DIR / "submission_audit.json").open(encoding="utf-8") as handle:
    audit = json.load(handle)
print(json.dumps(audit, ensure_ascii=False, indent=2))
print("READY:", OUTPUT_PATH)

## Output

검증이 끝난 제출 파일은 노트북과 같은 폴더의 `test_submission.csv`입니다. 재현 감사 정보는 `output/submission_audit.json`, 문항별 ORM 그룹과 폴백 정보는 `output/prediction_diagnostics.jsonl`에 남습니다.